In [2]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
txt_path = "household_power_consumption.txt"

if not os.path.exists(txt_path):
    print("завантаження архіву ")
    urllib.request.urlretrieve(url, zip_path)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
        
    print("файл household_power_consumption.txt успішно отримано.")
else:
    print("файл вже існує")

завантаження архіву 
файл household_power_consumption.txt успішно отримано.


In [3]:
# Блок завантажує датасет про споживання електроенергії, обробляючи роздільники ';' 
# та пропущені значення '?'. Виконується базове очищення даних (видалення рядків з NaN), 
# конвертація стовпців Date та Time у єдиний формат datetime, а також створення 
# нового категоріального атрибута (день тижня) для подальшого кодування.

import pandas as pd
import numpy as np
import timeit

def load_and_clean_power_data(filepath='household_power_consumption.txt'):
    df = pd.read_csv(filepath, sep=';', na_values=['?'], low_memory=False)
    df = df.dropna()
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
    df['DayOfWeek'] = df['Datetime'].dt.day_name()
    return df

print("Завантаження даних...")
df_power = load_and_clean_power_data()
print(f"Готово! Розмір датасету: {df_power.shape}")

Завантаження даних...
Готово! Розмір датасету: (2049280, 11)


In [4]:
# Блок містить чотири функції для формування складних вибірок згідно з умовами 
# завдання (фільтрація за потужністю, струмом, групами споживання та часом). 
# Також реалізовано профілювання швидкодії кожної процедури за допомогою модуля timeit, 
# що дозволяє оцінити часові витрати на виконання запитів до Pandas DataFrame.

def query_1(df):
    # вибірка: загальна активна потужність > 5 кВт
    return df[df['Global_active_power'] > 5.0]

def query_2(df):
    # вибірка: струм 19-20 А, Sub_2 (пральна/холод) > Sub_3 (бойлер/конд)
    return df[(df['Global_intensity'] >= 19) & 
              (df['Global_intensity'] <= 20) & 
              (df['Sub_metering_2'] > df['Sub_metering_3'])]

def query_3(df):
    # вибірка: 500 000 випадкових записів, розрахунок середнього для 3-х груп
    sample = df.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def query_4(df):
    # вибірка: після 18:00, потужність > 6 кВт, Sub_2 найбільша. 
    # кожен 3-й з першої половини та кожен 4-й з другої.
    mask = (df['Datetime'].dt.hour >= 18) & (df['Global_active_power'] > 6.0) & \
           (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])
    filtered = df[mask].reset_index(drop=True)
    
    half = len(filtered) // 2
    res1 = filtered.iloc[:half:3]
    res2 = filtered.iloc[half::4]
    return pd.concat([res1, res2])

print("Профілювання часу виконання (в секундах):")
print(f"Вибірка 1: {timeit.timeit(lambda: query_1(df_power), number=1):.4f} с")
print(f"Вибірка 2: {timeit.timeit(lambda: query_2(df_power), number=1):.4f} с")
print(f"Вибірка 3: {timeit.timeit(lambda: query_3(df_power), number=1):.4f} с")
print(f"Вибірка 4: {timeit.timeit(lambda: query_4(df_power), number=1):.4f} с")

Профілювання часу виконання (в секундах):
Вибірка 1: 0.0103 с
Вибірка 2: 0.0148 с
Вибірка 3: 0.3687 с
Вибірка 4: 0.0848 с


In [5]:
# Блок реалізує математичну підготовку даних: нормалізацію (приведення значень 
# до діапазону 0-1 за методом Min-Max) та стандартизацію (Z-оцінка, зміщення 
# до нульового середнього та одиничної дисперсії) для вибраного числового атрибута.

def normalize_and_standardize(df, column='Global_active_power'):
    col = df[column]
    # Min-Max нормалізація
    normalized = (col - col.min()) / (col.max() - col.min())
    # Z-score стандартизація
    standardized = (col - col.mean()) / col.std()
    return normalized, standardized

norm_data, stand_data = normalize_and_standardize(df_power)
print("Стандартизовані дані (перші 5 значень):")
print(stand_data.head())

Стандартизовані дані (перші 5 значень):
0    2.955076
1    4.037084
2    4.050325
3    4.063566
4    2.434881
Name: Global_active_power, dtype: float64


In [7]:
# Блок обчислює статистичні коефіцієнти кореляції Пірсона (оцінка лінійної 
# залежності) та Спірмена (оцінка монотонної рангової залежності) між двома 
# числовими атрибутами (активною потужністю та силою струму).

def calc_correlations(df, col1='Global_active_power', col2='Global_intensity'):
    pearson = df[col1].corr(df[col2], method='pearson')
    spearman = df[col1].corr(df[col2], method='spearman')
    
    print(f"Кореляція між {col1} та {col2}:")
    print(f"Пірсона: {pearson:.4f} | Спірмена: {spearman:.4f}")

calc_correlations(df_power)

Кореляція між Global_active_power та Global_intensity:
Пірсона: 0.9989 | Спірмена: 0.9954


In [ ]:
# Блок виконує One-Hot Encoding (OHE) для створеного категоріального атрибута 
# (день тижня). Процедура перетворює текстові класи на набір бінарних стовпців 
# (0 або 1) за допомогою функції pd.get_dummies, що необхідно для машинного навчання.

def perform_ohe(df, col='DayOfWeek'):
    # OHE 
    encoded_df = pd.get_dummies(df.head(500), columns=[col], dtype=int)
    return encoded_df

print("Результат One-Hot Encoding:")
perform_ohe(df_power).head()

Результат One-Hot Encoding:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,DayOfWeek_Saturday,DayOfWeek_Sunday
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,1,0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,1,0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,1,0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,1,0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,1,0
